# Part 3 – Machine Learning and AI

## Task 2 – Unsupervised Machine Learning

### Objective

This task applies unsupervised machine learning techniques to identify meaningful patterns in historical traffic conditions.

Two techniques are implemented:

1. **K-means clustering** – groups traffic observations according to time, weather severity and traffic volume.
2. **Association rule mining** – identifies combinations of time, day type and weather conditions that are associated with congestion levels.

The analysis uses the engineered traffic dataset produced in Part 2.

In [1]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score


# ------------------------------------------------------------
# Logging configuration
# ------------------------------------------------------------

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
logger.propagate = False

if logger.handlers:
    logger.handlers.clear()

formatter = logging.Formatter(
    "%(asctime)s - %(levelname)s - %(name)s - %(message)s"
)

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(formatter)

file_handler = logging.FileHandler(
    "task2_unsupervised_ml.log",
    mode="w"
)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

logger.info(
    "Task 2 unsupervised machine learning workflow started."
)

2026-10-06 23:48:23,016 - INFO - __main__ - Task 2 unsupervised machine learning workflow started.


## 2.3 Load the Engineered Traffic Dataset

The engineered dataset created in Part 2 is reused as the input for unsupervised learning.

K-means clustering is applied to traffic conditions rather than accident records because no real accident dataset was supplied.


In [2]:
DATA_PATH = Path(
    "../part2_python/engineered_traffic_data.csv"
)

try:
    df = pd.read_csv(
        DATA_PATH,
        parse_dates=["date_time"]
    )

    logger.info(
        "Engineered traffic dataset loaded successfully: "
        "%d rows, %d columns.",
        df.shape[0],
        df.shape[1]
    )

except FileNotFoundError:
    logger.error(
        "Engineered traffic dataset not found: %s",
        DATA_PATH
    )
    raise

except (pd.errors.ParserError, ValueError) as error:
    logger.error(
        "Unable to load engineered traffic dataset: %s",
        error
    )
    raise

df.shape

2026-10-06 23:48:23,389 - INFO - __main__ - Engineered traffic dataset loaded successfully: 48187 rows, 30 columns.


(48187, 30)

## 2.4 Prepare Features for K-Means Clustering

K-means clustering is used to group historical traffic conditions.

Three interpretable variables are used:

- `hour` – time of day
- `weather_severity` – numerical representation of weather conditions
- `traffic_volume` – observed traffic volume

Weather severity is encoded into three levels:

- 0 = normal weather
- 1 = reduced-visibility or moderate weather
- 2 = severe weather

The variables are standardised before clustering because they are measured on different numerical scales.



In [3]:
# ------------------------------------------------------------
# Task 2.4 - Prepare features for K-means clustering
# ------------------------------------------------------------

MODERATE_WEATHER = {
    "Rain",
    "Drizzle",
    "Fog",
    "Mist",
    "Haze",
    "Smoke"
}

SEVERE_WEATHER = {
    "Thunderstorm",
    "Snow",
    "Squall"
}


def encode_weather_severity(weather):
    """Convert weather condition into an ordinal severity level."""

    if weather in SEVERE_WEATHER:
        return 2

    elif weather in MODERATE_WEATHER:
        return 1

    else:
        return 0


# ------------------------------------------------------------
# Create weather-severity feature
# ------------------------------------------------------------

df["weather_severity"] = (
    df["weather_main"]
    .apply(encode_weather_severity)
)


# ------------------------------------------------------------
# Define clustering features
# ------------------------------------------------------------

cluster_features = [
    "hour",
    "weather_severity",
    "traffic_volume"
]


# ------------------------------------------------------------
# Create clustering dataset
# ------------------------------------------------------------

X_cluster = df[
    cluster_features
].copy()


# ------------------------------------------------------------
# Validate clustering dataset
# ------------------------------------------------------------

if X_cluster.isnull().any().any():

    logger.error(
        "Missing values detected in clustering features."
    )

    raise ValueError(
        "Clustering features contain missing values."
    )


# ------------------------------------------------------------
# Standardise features
# ------------------------------------------------------------

cluster_scaler = StandardScaler()

X_cluster_scaled = (
    cluster_scaler.fit_transform(
        X_cluster
    )
)


logger.info(
    "K-means features prepared successfully: %s",
    cluster_features
)


# ------------------------------------------------------------
# User-facing validation output
# ------------------------------------------------------------

print(
    "Clustering dataset shape:",
    X_cluster.shape
)

print(
    "\nWeather severity distribution:"
)

print(
    df["weather_severity"]
    .value_counts()
    .sort_index()
)

print(
    "\nClustering feature summary:"
)

print(
    X_cluster
    .describe()
    .round(2)
)

2026-10-06 23:48:23,547 - INFO - __main__ - K-means features prepared successfully: ['hour', 'weather_severity', 'traffic_volume']


Clustering dataset shape: (48187, 3)

Weather severity distribution:
weather_severity
0    28542
1    15733
2     3912
Name: count, dtype: int64

Clustering feature summary:
           hour  weather_severity  traffic_volume
count  48187.00          48187.00        48187.00
mean      11.40              0.49         3259.62
std        6.94              0.64         1986.95
min        0.00              0.00            0.00
25%        5.00              0.00         1192.50
50%       11.00              0.00         3379.00
75%       17.00              1.00         4933.00
max       23.00              2.00         7280.00


## 2.5 Select the Number of Clusters

Candidate values of K from 2 to 6 are compared using the silhouette score.

Because silhouette calculation can be computationally expensive on the full dataset, a reproducible sample is used for model selection. The final K-means model is then trained on the complete dataset.

A higher silhouette score indicates better separation between clusters.

In [4]:
# ------------------------------------------------------------
# Reproducible sample for silhouette evaluation
# ------------------------------------------------------------

rng = np.random.RandomState(42)

sample_size = min(
    10000,
    len(X_cluster_scaled)
)

sample_indices = rng.choice(
    len(X_cluster_scaled),
    size=sample_size,
    replace=False
)

X_silhouette = X_cluster_scaled[
    sample_indices
]


# ------------------------------------------------------------
# Evaluate K = 2 to 6
# ------------------------------------------------------------

silhouette_results = []

for k in range(2, 7):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(
        X_silhouette
    )

    score = silhouette_score(
        X_silhouette,
        labels
    )

    silhouette_results.append(
        {
            "K": k,
            "Silhouette_Score": score
        }
    )

    logger.info(
        "K=%d silhouette score=%.4f",
        k,
        score
    )


silhouette_df = pd.DataFrame(
    silhouette_results
)

silhouette_df.round(4)

2026-10-06 23:48:32,429 - INFO - __main__ - K=2 silhouette score=0.4241
2026-10-06 23:48:37,807 - INFO - __main__ - K=3 silhouette score=0.4404
2026-10-06 23:48:42,126 - INFO - __main__ - K=4 silhouette score=0.4623
2026-10-06 23:48:46,610 - INFO - __main__ - K=5 silhouette score=0.4997
2026-10-06 23:48:51,135 - INFO - __main__ - K=6 silhouette score=0.5254


,K,Silhouette_Score
0,2,0.4241
1,3,0.4404
2,4,0.4623
3,5,0.4997
4,6,0.5254


## 2.6 Final K-Means Clustering Model

Candidate values from K=2 to K=6 were evaluated using the silhouette score.

The highest silhouette score was obtained for **K=6 (0.5254)**. Therefore, six clusters are used for the final K-means model.

The final model is trained using the complete standardised clustering dataset rather than the sample used during model selection.


In [5]:
# ------------------------------------------------------------
# Task 2.6 - Final K-means model
# ------------------------------------------------------------

best_k = int(
    silhouette_df.loc[
        silhouette_df["Silhouette_Score"].idxmax(),
        "K"
    ]
)

best_silhouette = float(
    silhouette_df["Silhouette_Score"].max()
)


final_kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["traffic_cluster"] = (
    final_kmeans.fit_predict(
        X_cluster_scaled
    )
)


logger.info(
    "Final K-means model trained successfully with K=%d.",
    best_k
)


print("Selected K:", best_k)

print(
    "Best silhouette score:",
    round(best_silhouette, 4)
)

print("\nCluster sizes:")

print(
    df["traffic_cluster"]
    .value_counts()
    .sort_index()
)

2026-10-06 23:48:51,608 - INFO - __main__ - Final K-means model trained successfully with K=6.


Selected K: 6
Best silhouette score: 0.5254

Cluster sizes:
traffic_cluster
0     8814
1     7062
2     7847
3     4281
4     6550
5    13633
Name: count, dtype: int64


## 2.7 Cluster Interpretation

The six clusters are summarised using their average hour, weather severity and traffic volume.

The cluster numbers themselves do not have an inherent meaning. Their interpretation is based on the observed characteristics of the records assigned to each cluster.

In [6]:
# ------------------------------------------------------------
# Task 2.7 - Cluster summary
# ------------------------------------------------------------

cluster_summary = (
    df.groupby("traffic_cluster")
    .agg(
        average_hour=("hour", "mean"),
        average_weather_severity=(
            "weather_severity",
            "mean"
        ),
        average_traffic_volume=(
            "traffic_volume",
            "mean"
        ),
        observations=(
            "traffic_volume",
            "size"
        )
    )
    .round(2)
)


logger.info(
    "K-means cluster summary created successfully."
)

cluster_summary

2026-10-06 23:48:51,711 - INFO - __main__ - K-means cluster summary created successfully.


,average_hour,average_weather_severity,average_traffic_volume,observations
traffic_cluster,,,,
0,11.42,1.19,5038.20,8814
1,20.62,0.00,2670.39,7062
2,3.13,0.00,938.15,7847
3,20.46,1.23,2513.60,4281
4,3.32,1.19,941.37,6550
5,12.40,0.00,5099.24,13633


The six clusters form three time-of-day profiles, each split by weather severity. Clusters 2 and 4 are overnight low-traffic conditions, with mean volumes of about 938 and 941 vehicles. Cluster 2 occurs in normal weather, while Cluster 4 occurs in moderate or severe weather. Clusters 1 and 3 are evening wind-down conditions, around 20:30, with mean volumes of 2,670 and 2,514. Again, the split is weather rather than traffic level. Clusters 5 and 0 are daytime congested conditions, with mean volumes of 5,099 and 5,038. Cluster 5 is normal weather and Cluster 0 is heavier weather. Traffic volume is therefore driven mainly by hour. Weather separates otherwise similar hours but does not create a separate congestion level. This agrees with the weak temperature correlation found in Part 1. The cluster labels are descriptive groupings, not predictions of accidents.

In [7]:
import mlxtend

print(
    "mlxtend version:",
    mlxtend.__version__
)


mlxtend version: 0.25.0


In [8]:
from mlxtend.frequent_patterns import (
    apriori,
    association_rules
)

logger.info(
    "Association-rule mining libraries loaded successfully."
)

2026-10-06 23:51:13,719 - INFO - __main__ - Association-rule mining libraries loaded successfully.


## 2.8 Prepare Data for Association Rule Mining

Association rule mining is used to identify combinations of historical traffic conditions that are associated with congestion levels.

The dataset is converted into transaction-style categorical features representing:

- Time of day
- Weekday or weekend
- Weather condition
- Congestion level

Time of day is divided into Night, Morning, Afternoon and Evening periods. Day type is represented as Weekday or Weekend, while the existing weather and congestion categories are converted into transaction items.

The resulting one-hot encoded transaction dataset is used as input to the Apriori algorithm.



In [9]:
# ------------------------------------------------------------
# Task 2.8 - Prepare association-rule transactions
# ------------------------------------------------------------

association_df = df.copy()


# ------------------------------------------------------------
# Time-of-day categories
# ------------------------------------------------------------

def assign_time_period(hour):
    if 0 <= hour <= 5:
        return "Time_Night"
    elif 6 <= hour <= 11:
        return "Time_Morning"
    elif 12 <= hour <= 17:
        return "Time_Afternoon"
    else:
        return "Time_Evening"


association_df["time_period"] = (
    association_df["hour"]
    .apply(assign_time_period)
)


# ------------------------------------------------------------
# Day-type categories
# ------------------------------------------------------------

association_df["day_type"] = np.where(
    association_df["is_weekend"] == 1,
    "Day_Weekend",
    "Day_Weekday"
)


# ------------------------------------------------------------
# Weather categories
# ------------------------------------------------------------

association_df["weather_item"] = (
    "Weather_"
    + association_df["weather_main"]
    .fillna("Unknown")
    .astype(str)
)


# ------------------------------------------------------------
# Congestion categories
# ------------------------------------------------------------

association_df["congestion_item"] = (
    "Congestion_"
    + association_df["congestion_category"]
    .astype(str)
)


# ------------------------------------------------------------
# Convert observations into one-hot transactions
# ------------------------------------------------------------

transaction_columns = [
    "time_period",
    "day_type",
    "weather_item",
    "congestion_item"
]

transactions = pd.get_dummies(
    association_df[
        transaction_columns
    ],
    prefix="",
    prefix_sep=""
).astype(bool)


logger.info(
    "Association-rule transaction dataset created: "
    "%d rows, %d items.",
    transactions.shape[0],
    transactions.shape[1]
)


print(
    "Transaction dataset shape:",
    transactions.shape
)

print(
    "\nExample transaction items:"
)

print(
    transactions.columns.tolist()
)

2026-10-06 23:52:23,403 - INFO - __main__ - Association-rule transaction dataset created: 48187 rows, 20 items.


Transaction dataset shape: (48187, 20)

Example transaction items:
['Time_Afternoon', 'Time_Evening', 'Time_Morning', 'Time_Night', 'Day_Weekday', 'Day_Weekend', 'Weather_Clear', 'Weather_Clouds', 'Weather_Drizzle', 'Weather_Fog', 'Weather_Haze', 'Weather_Mist', 'Weather_Rain', 'Weather_Smoke', 'Weather_Snow', 'Weather_Squall', 'Weather_Thunderstorm', 'Congestion_High', 'Congestion_Low', 'Congestion_Medium']


## 2.9 Mine Association Rules

Frequent itemsets are generated using the Apriori algorithm.

Rules are restricted to those where the consequent contains a congestion category because the objective is to identify conditions associated with congestion.

The strongest rules are ranked by lift and interpreted together with support and confidence.

In [11]:
# ------------------------------------------------------------
# Task 2.9 - Frequent itemsets
# ------------------------------------------------------------

frequent_itemsets = apriori(
    transactions,
    min_support=0.02,
    use_colnames=True
)


logger.info(
    "Apriori generated %d frequent itemsets.",
    len(frequent_itemsets)
)


# ------------------------------------------------------------
# Generate association rules
# ------------------------------------------------------------

rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)


# ------------------------------------------------------------
# Keep rules predicting congestion
# ------------------------------------------------------------

def contains_congestion(itemset):
    return any(
        str(item).startswith("Congestion_")
        for item in itemset
    )


congestion_rules = rules.loc[
    rules["consequents"]
    .apply(contains_congestion)
].copy()


# ------------------------------------------------------------
# Exclude congestion from antecedents
# ------------------------------------------------------------

congestion_rules = congestion_rules.loc[
    ~congestion_rules["antecedents"]
    .apply(contains_congestion)
].copy()


# ------------------------------------------------------------
# Rank by lift
# ------------------------------------------------------------

top_rules = (
    congestion_rules
    .sort_values(
        ["lift", "confidence", "support"],
        ascending=False
    )
    .head(10)
    .copy()
)


logger.info(
    "Association-rule mining completed: "
    "%d congestion rules retained.",
    len(congestion_rules)
)


# ------------------------------------------------------------
# Make rules readable
# ------------------------------------------------------------

def itemset_to_text(itemset):
    return ", ".join(
        sorted(str(item) for item in itemset)
    )


rule_display = top_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].copy()

rule_display["antecedents"] = (
    rule_display["antecedents"]
    .apply(itemset_to_text)
)

rule_display["consequents"] = (
    rule_display["consequents"]
    .apply(itemset_to_text)
)

rule_display = rule_display.reset_index(
    drop=True
)

rule_display.round(4)

2026-10-06 23:54:12,682 - INFO - __main__ - Apriori generated 154 frequent itemsets.
2026-10-06 23:54:12,713 - INFO - __main__ - Association-rule mining completed: 184 congestion rules retained.


,antecedents,consequents,support,confidence,lift
0,"Day_Weekend, Time_Night","Congestion_Low, Weather_Clear",0.0220,0.3023,3.8336
1,"Time_Night, Weather_Clouds","Congestion_Low, Day_Weekday",0.0355,0.6143,3.6352
2,"Day_Weekend, Time_Night",Congestion_Low,0.0644,0.8861,3.5443
3,Time_Night,"Congestion_Low, Day_Weekday, Weather_Clouds",0.0355,0.1392,3.5423
4,Time_Night,"Congestion_Low, Day_Weekday, Weather_Clear",0.0476,0.1868,3.5409
5,Time_Night,"Congestion_Low, Day_Weekday, Weather_Mist",0.0244,0.0959,3.5337
6,Time_Night,"Congestion_Low, Day_Weekday",0.1519,0.5960,3.5271
7,"Day_Weekday, Time_Afternoon","Congestion_High, Weather_Clouds",0.0559,0.3224,3.5196
8,Time_Night,"Congestion_Low, Weather_Clear",0.0696,0.2730,3.4622
9,"Time_Night, Weather_Mist","Congestion_Low, Day_Weekday",0.0244,0.5835,3.4527


## 2.10 Association Rule Findings and Interpretation

The association rules were ranked by lift. A lift greater than 1 indicates that the conditions in the antecedent and consequent occur together more frequently than would be expected if they were independent.

The strongest rules reveal clear relationships between time of day, day type, weather and congestion.

### Key Finding 1 – Weekend nights are strongly associated with low congestion

The rule:

**Weekend + Night → Low Congestion**

has:

- Support = 0.0644
- Confidence = 0.8861
- Lift = 3.5443

This means that approximately 88.6% of records matching weekend-night conditions were associated with low congestion. The lift of 3.54 indicates that low congestion occurs substantially more often under these conditions than would be expected by chance.

In practical terms, historical weekend-night travel is strongly associated with lower traffic congestion.

### Key Finding 2 – Night-time weekday traffic is also associated with low congestion

The rule:

**Night → Low Congestion + Weekday**

has:

- Support = 0.1519
- Confidence = 0.5960
- Lift = 3.5271

This suggests that night-time observations frequently combine weekday conditions with low congestion. The relatively high support indicates that this is not an isolated pattern in the dataset.

Several related rules involving clear, cloudy and mist weather also associate night-time travel with low congestion.

### Key Finding 3 – Weekday afternoons are associated with higher congestion

The rule:

**Weekday + Afternoon → High Congestion + Cloudy Weather**

has:

- Support = 0.0559
- Confidence = 0.3224
- Lift = 3.5196

Although its confidence is lower than the weekend-night rule, its lift indicates a meaningful association. This suggests that weekday afternoon conditions are more strongly associated with high congestion than would be expected if these conditions were independent.

### Overall Interpretation

The highest-lift rules indicate that **time of day and day type are important factors in historical congestion patterns**.

Night-time travel, particularly on weekends, is strongly associated with low congestion. In contrast, weekday afternoon conditions appear among the strongest rules associated with higher congestion.

These rules describe historical associations and should not be interpreted as causal relationships or real-time traffic forecasts.


In [15]:
# ------------------------------------------------------------
# Task 2.11 - Final validation
# ------------------------------------------------------------

task2_validation = {
    "K-means model trained":
        "traffic_cluster" in df.columns,

    "K selection evaluated":
        len(silhouette_df) == 5,

    "Six clusters created":
        df["traffic_cluster"].nunique() == 6,

    "All clusters summarised":
        len(cluster_summary) == 6,

    "Association rules generated":
        len(congestion_rules) > 0,

    "Top rules reported":
        len(rule_display) > 0,

    "Highest lift greater than 1":
        rule_display["lift"].max() > 1
}


logger.info(
    "Task 2 final validation completed."
)


print("TASK 2 FINAL VALIDATION")
print("-" * 50)

for check, passed in task2_validation.items():
    status = "PASS" if passed else "FAIL"
    print(f"{check}: {status}")

2026-10-07 00:06:39,135 - INFO - __main__ - Task 2 final validation completed.


TASK 2 FINAL VALIDATION
--------------------------------------------------
K-means model trained: PASS
K selection evaluated: PASS
Six clusters created: PASS
All clusters summarised: PASS
Association rules generated: PASS
Top rules reported: PASS
Highest lift greater than 1: PASS


## 2.12 Task 2 Summary

Task 2 applied two unsupervised machine learning techniques to the engineered traffic dataset.

### K-Means Clustering

Traffic conditions were clustered using hour, weather severity and traffic volume. Candidate values from K=2 to K=6 were evaluated using the silhouette score.

K=6 achieved the highest silhouette score of **0.5254** and was selected for the final clustering model.

The resulting clusters provide a data-driven way to identify different historical traffic-condition profiles based on time, weather severity and traffic volume.

### Association Rule Mining

Association rule mining was applied after converting time of day, day type, weather and congestion into transaction-style categorical features.

The highest-lift rules revealed meaningful historical patterns. In particular, weekend-night conditions were strongly associated with low congestion, with a confidence of **0.8861** and lift of **3.5443**.

Night-time conditions more generally appeared repeatedly among the strongest low-congestion associations, while weekday-afternoon conditions appeared among the strongest associations involving higher congestion.

These findings provide useful evidence for the travel-timing recommendation system developed later in the capstone.

The identified patterns represent historical associations, not causal effects or real-time forecasts. They support the later recommendation task: weekend and weeknight periods are the strongest low-congestion travel windows, while weekday afternoons are the periods most often associated with higher congestion.